# RAG Ingestion Development Notebook
### germ//clone — Meng Hai's ingestion pipeline (transcripts + textbooks)

**Purpose:** This notebook is your interactive workbench for designing and tuning the ingestion pipeline.  
You will load raw source files, chunk them, inspect the results, and adjust parameters — all without writing to the database.  
At the end, you export the approved chunks to a JSON file ready for the Phase 2 upsert script.

**Run each section top to bottom.** Every code cell has an explanation above it.  
Parameters you can tune are clearly marked with `# ← TUNE THIS`.

---
**Requirements before running:**
- Backend venv activated or VS Code pointing to `source/backend/.venv`
- `source/backend/.env` contains `GEMINI_API_KEY` and `DATABASE_URL`
- Corpus files in `corpus/` at repo root (`lesson01.json`, both PDFs)

---
## Section 0 — What is RAG Ingestion?

RAG stands for **Retrieval-Augmented Generation**. It is a technique that makes an AI answer questions using *your specific documents* rather than just its training data.

There are three stages:

```
┌─────────────────────────────────────────────────────────────────┐
│  STAGE 1: INGESTION  (this notebook)                            │
│                                                                 │
│  Raw files → chunks → embeddings → stored in Postgres/pgvector │
│                                                                 │
│  STAGE 2: RETRIEVAL  (queries.py)                               │
│                                                                 │
│  Student asks a question → embed question → find the most       │
│  similar chunks in the DB → return top-K chunks                 │
│                                                                 │
│  STAGE 3: GENERATION  (compose.py)                              │
│                                                                 │
│  Retrieved chunks + question → LLM → grounded answer           │
└─────────────────────────────────────────────────────────────────┘
```

### What is a "chunk"?
A chunk is a short, self-contained piece of text from your source material — usually 100–400 words.  
Why not just store the whole document? Because:
- A 200-page textbook is too long to pass to an LLM in one go
- Smaller chunks give more precise retrieval (you match the *exact* relevant passage)
- Each chunk gets embedded as a single vector — you need manageable units

### What makes a good chunk?
- Enough context to be understood on its own (too short = missing context)
- Not so long it becomes unfocused (too long = embedding averages over too many topics)
- The **sweet spot** is roughly 150–350 words

---
## Setup — Imports and Paths

This cell sets up all the libraries and file paths we'll use throughout the notebook.  
Run this first — every other section depends on it.

In [ ]:
import json
import os
import sys
import time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from dotenv import load_dotenv

# --- Paths (relative to repo root) ---
REPO_ROOT   = Path("../").resolve()
CORPUS_DIR  = REPO_ROOT / "corpus"
BACKEND_DIR = REPO_ROOT / "source" / "backend"
OUTPUT_DIR  = Path(".").resolve()   # notebooks/ — chunks_preview.json saved here

# Add backend to sys.path so we can import ingestion helpers
sys.path.insert(0, str(BACKEND_DIR))

# Load .env (contains GEMINI_API_KEY, DATABASE_URL, etc.)
load_dotenv(BACKEND_DIR / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
DATABASE_URL   = os.getenv("DATABASE_URL")

# Verify
print(f"CORPUS_DIR  : {CORPUS_DIR}")
print(f"BACKEND_DIR : {BACKEND_DIR}")
print(f"GEMINI_API_KEY loaded : {'✓' if GEMINI_API_KEY else '✗  check source/backend/.env'}")
print(f"DATABASE_URL loaded   : {'✓' if DATABASE_URL else '✗  alignment section will be skipped'}")
print()
print("Corpus files found:")
for f in sorted(CORPUS_DIR.iterdir()):
    if f.suffix in (".json", ".pdf"):
        size_mb = f.stat().st_size / 1_048_576
        print(f"  {f.name:<45}  {size_mb:5.1f} MB")

---
## Section 1 — Load and Inspect the Transcript

The transcript file (`lesson01.json`) was produced by Whisper ASR with speaker diarization.  
Diarization means the tool automatically detected *who* was speaking and labelled each speaker `SPEAKER_XX`.

Each **segment** in the JSON has four fields:
- `text` — the transcribed speech (raw, may have grammar quirks)
- `start` / `end` — when this segment occurred, in **seconds** from the start of the recording
- `avg_logprob` — confidence score from the ASR model (closer to 0 = more confident; below -0.5 = low confidence)
- `speaker` — who said it, e.g. `SPEAKER_17`

Let's load the file and see what we're working with.

In [ ]:
# --- Load the transcript ---
TRANSCRIPT_FILE = CORPUS_DIR / "lesson01.json"   # ← change to any lessonNN.json

with open(TRANSCRIPT_FILE) as f:
    transcript = json.load(f)

segments = transcript["segments"]
language = transcript.get("language", "unknown")

total_duration_min = max(s["end"] for s in segments) / 60
total_words = sum(len(s["text"].split()) for s in segments)

print(f"File           : {TRANSCRIPT_FILE.name}")
print(f"Language       : {language}")
print(f"Segments       : {len(segments)}")
print(f"Duration       : {total_duration_min:.0f} minutes")
print(f"Total words    : {total_words:,}")
print(f"Avg words/seg  : {total_words / len(segments):.0f}")
print(f"Unique speakers: {len(set(s['speaker'] for s in segments))}")

# Show the first 5 segments as a table
print("\n--- First 5 segments ---")
df_sample = pd.DataFrame([
    {
        "speaker": s["speaker"],
        "start (min)": f"{s['start']/60:.1f}",
        "end (min)":   f"{s['end']/60:.1f}",
        "confidence":  f"{s['avg_logprob']:.2f}",
        "text (first 80 chars)": s["text"].strip()[:80]
    }
    for s in segments[:5]
])
display(df_sample)

In [ ]:
# --- Timeline chart: who spoke when ---
# Each speaker gets a colour; bars show when they were active.
# This helps you visually see the lecture flow.

speakers = sorted(set(s["speaker"] for s in segments))
colours   = plt.cm.tab20(np.linspace(0, 1, len(speakers)))
sp_colour = {sp: colours[i] for i, sp in enumerate(speakers)}

fig, ax = plt.subplots(figsize=(14, 3))
for seg in segments:
    ax.barh(
        seg["speaker"],
        seg["end"] - seg["start"],
        left=seg["start"] / 60,
        color=sp_colour[seg["speaker"]],
        height=0.6,
    )

ax.set_xlabel("Time (minutes)")
ax.set_title(f"Speaker timeline — {TRANSCRIPT_FILE.name}")
ax.invert_yaxis()
plt.tight_layout()
plt.show()
print("The speaker who dominates the chart is the instructor.")

---
## Section 2 — Identify the Instructor

With 13 speakers in the recording, we need to know which one is the instructor.  
The simplest reliable method: **the instructor speaks the most words**.  

We count the total words per speaker and rank them. The top speaker becomes the instructor.  
All other speakers are treated as students.

> If you already know the instructor's speaker label (e.g. you were in the meeting), you can hardcode it below.

In [ ]:
# --- Count words per speaker ---
speaker_words = defaultdict(int)
speaker_segs  = defaultdict(int)
for seg in segments:
    speaker_words[seg["speaker"]] += len(seg["text"].split())
    speaker_segs[seg["speaker"]]  += 1

# Auto-detect instructor
INSTRUCTOR = max(speaker_words, key=speaker_words.get)  # ← TUNE THIS if auto-detect is wrong

# Display ranked table
df_speakers = pd.DataFrame([
    {
        "speaker": sp,
        "total words": speaker_words[sp],
        "% of total": f"{100 * speaker_words[sp] / total_words:.1f}%",
        "segments": speaker_segs[sp],
        "role": "INSTRUCTOR ←" if sp == INSTRUCTOR else "student"
    }
    for sp in sorted(speaker_words, key=speaker_words.get, reverse=True)
])
display(df_speakers)

print(f"\nAuto-detected instructor: {INSTRUCTOR} ({speaker_words[INSTRUCTOR]:,} words)")
print("If this is wrong, change the INSTRUCTOR variable above and re-run.")

In [ ]:
# --- Verify by sampling instructor segments ---
# Read 3 random instructor segments to confirm they look like lecture content.
import random
random.seed(42)
instructor_segs = [s for s in segments if s["speaker"] == INSTRUCTOR]
samples = random.sample(instructor_segs, min(3, len(instructor_segs)))
print(f"Sample instructor segments ({INSTRUCTOR}):")
print()
for i, s in enumerate(samples, 1):
    mins = s["start"] / 60
    print(f"  [{i}] @ {mins:.0f} min: {s['text'].strip()[:200]}")
    print()

---
## Section 3 — Chunking the Transcript Into Turns

A single Whisper segment is typically one or two sentences — too short for meaningful retrieval.  
We merge consecutive segments **from the same speaker** into a longer unit called a **turn**.

A turn ends when:
1. The speaker changes (another person starts talking), or
2. The turn exceeds `CHUNK_MAX_WORDS` (to avoid overly long chunks)

Short student turns (< `MIN_TURN_WORDS`) are merged into the instructor turn that **follows** them  
— this preserves Q&A context ("student asks → instructor answers").

Turns with fewer than `MIN_TURN_WORDS` that can't be merged are **dropped** (noise: greetings, "okay", "thanks").

In [ ]:
# ============================================================
# TUNABLE PARAMETERS  (adjust and re-run to see the effect)
# ============================================================
CHUNK_MAX_WORDS = 300    # ← TUNE THIS: max words per turn before splitting
MIN_TURN_WORDS  = 20     # ← TUNE THIS: turns shorter than this are dropped/merged
# ============================================================

def chunk_transcript(segs, instructor_id, max_words, min_words):
    """
    Merge consecutive same-speaker segments into turns.
    Returns a list of turn dicts.
    """
    turns = []

    # Step 1: merge consecutive same-speaker segments
    raw_turns = []
    current = None
    for seg in segs:
        if current is None or seg["speaker"] != current["speaker"]:
            if current:
                raw_turns.append(current)
            current = {
                "speaker":    seg["speaker"],
                "text":       seg["text"],
                "start":      seg["start"],
                "end":        seg["end"],
                "confidence": seg["avg_logprob"],
            }
        else:
            # Split if this segment would push us over max_words
            combined = current["text"] + seg["text"]
            if len(combined.split()) > max_words:
                raw_turns.append(current)
                current = {
                    "speaker":    seg["speaker"],
                    "text":       seg["text"],
                    "start":      seg["start"],
                    "end":        seg["end"],
                    "confidence": seg["avg_logprob"],
                }
            else:
                current["text"] += seg["text"]
                current["end"]   = seg["end"]
                current["confidence"] = max(current["confidence"], seg["avg_logprob"])
    if current:
        raw_turns.append(current)

    # Step 2: merge short student questions into the following instructor turn
    merged_turns = []
    i = 0
    while i < len(raw_turns):
        turn = raw_turns[i]
        words = len(turn["text"].split())
        is_short_student = (turn["speaker"] != instructor_id and words < min_words)
        # If short student turn AND next turn is instructor → merge
        if (is_short_student
                and i + 1 < len(raw_turns)
                and raw_turns[i + 1]["speaker"] == instructor_id):
            nxt = raw_turns[i + 1]
            merged = {
                "speaker":    instructor_id,
                "text":       f"[Student: {turn['text'].strip()}] {nxt['text']}",
                "start":      turn["start"],
                "end":        nxt["end"],
                "confidence": nxt["confidence"],
                "source_type": "transcript_qa",   # Q&A pair
            }
            merged_turns.append(merged)
            i += 2  # skip the next turn (already merged)
        else:
            merged_turns.append(turn)
            i += 1

    # Step 3: drop remaining short turns (noise)
    kept, dropped = [], []
    for turn in merged_turns:
        words = len(turn["text"].split())
        if words >= min_words:
            kept.append(turn)
        else:
            dropped.append(turn)

    # Step 4: assign source_type and index
    stem = TRANSCRIPT_FILE.stem  # e.g. "lesson01"
    for i, turn in enumerate(kept):
        if "source_type" not in turn:
            turn["source_type"] = (
                "transcript_instructor"
                if turn["speaker"] == instructor_id
                else "transcript_student"
            )
        turn["chunk_id"]   = f"mh_tr_{stem}_{i:04d}"
        turn["source_file"] = TRANSCRIPT_FILE.name

    return kept, dropped


turns, dropped_turns = chunk_transcript(
    segments, INSTRUCTOR, CHUNK_MAX_WORDS, MIN_TURN_WORDS
)

print(f"Original segments : {len(segments)}")
print(f"Turns kept        : {len(turns)}")
print(f"Turns dropped     : {len(dropped_turns)}  (noise)")
print()
type_counts = defaultdict(int)
for t in turns:
    type_counts[t["source_type"]] += 1
for k, v in sorted(type_counts.items()):
    print(f"  {k:<30}: {v} turns")

In [ ]:
# --- Preview: show a few turns ---
print("Sample instructor turn:")
t = next(t for t in turns if t["source_type"] == "transcript_instructor")
print(f"  chunk_id    : {t['chunk_id']}")
print(f"  timestamp   : {t['start']/60:.1f} – {t['end']/60:.1f} min")
print(f"  word count  : {len(t['text'].split())}")
print(f"  text        : {t['text'].strip()[:300]}")
print()
qa = next((t for t in turns if t["source_type"] == "transcript_qa"), None)
if qa:
    print("Sample Q&A turn (student question + instructor answer merged):")
    print(f"  chunk_id    : {qa['chunk_id']}")
    print(f"  text        : {qa['text'].strip()[:300]}")
print()
print("Sample dropped turns (should look like noise):")
for d in dropped_turns[:5]:
    print(f"  [{d['speaker']}] {d['text'].strip()[:80]}")

---
## Section 4 — Tuning the Chunking Parameters

The two parameters `CHUNK_MAX_WORDS` and `MIN_TURN_WORDS` control chunk quality.  

**What to look for in the histogram:**
- Most turns should cluster between 50–300 words
- Very few turns should be at the hard cap (300 words) — if many are hitting it, lower the cap or the turns are naturally long
- The dropped turns should all look like noise — if real content is being dropped, raise `MIN_TURN_WORDS`

Go back to Section 3, adjust `CHUNK_MAX_WORDS` and `MIN_TURN_WORDS`, and re-run to compare.

In [ ]:
# --- Histogram of turn word counts ---
word_counts = [len(t["text"].split()) for t in turns]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: histogram
axes[0].hist(word_counts, bins=30, color="steelblue", edgecolor="white")
axes[0].axvline(CHUNK_MAX_WORDS, color="red", linestyle="--", label=f"CHUNK_MAX_WORDS={CHUNK_MAX_WORDS}")
axes[0].axvline(MIN_TURN_WORDS,  color="orange", linestyle="--", label=f"MIN_TURN_WORDS={MIN_TURN_WORDS}")
axes[0].set_xlabel("Words per turn")
axes[0].set_ylabel("Count")
axes[0].set_title("Turn word count distribution")
axes[0].legend()

# Right: pie chart of source types
labels  = list(type_counts.keys())
sizes   = [type_counts[k] for k in labels]
axes[1].pie(sizes, labels=labels, autopct="%1.0f%%", startangle=90,
            colors=["#4C72B0", "#DD8452", "#55A868"])
axes[1].set_title("Turn type breakdown")

plt.suptitle(f"Chunking result: {len(turns)} turns kept, {len(dropped_turns)} dropped")
plt.tight_layout()
plt.show()

print(f"Median turn length : {int(np.median(word_counts))} words")
print(f"Mean turn length   : {int(np.mean(word_counts))} words")
print(f"Turns at max cap   : {sum(1 for w in word_counts if w >= CHUNK_MAX_WORDS - 5)} (should be small)")

---
## Section 5 — Load and Inspect a PDF

We have two textbooks in `corpus/`:  
- *Mathematics for Machine Learning* (Deisenroth et al.)  
- *Deep Learning* (Goodfellow, Bengio, Courville — MIT 2017)

We use **PyMuPDF** (imported as `fitz`) to extract text page by page.  
Math textbooks contain equations that may look garbled in plain text — this is normal.  
The embedding model captures the semantic meaning regardless of the exact character representation.

In [ ]:
import fitz   # PyMuPDF

# ← TUNE THIS: change to inspect the other textbook
PDF_FILE = CORPUS_DIR / "Mathematics_for_Machine_Learning.pdf"

doc = fitz.open(PDF_FILE)

print(f"File        : {PDF_FILE.name}")
print(f"Pages       : {len(doc)}")
print(f"Size        : {PDF_FILE.stat().st_size / 1_048_576:.1f} MB")
print()

# Show text from a few sample pages
for page_num in [10, 50, 100]:
    if page_num < len(doc):
        page = doc[page_num]
        text = page.get_text().strip()
        word_count = len(text.split())
        print(f"--- Page {page_num} ({word_count} words) ---")
        print(text[:400])
        print()

---
## Section 6 — Chunking the PDF

PDF text is extracted page by page.  
We merge consecutive pages into chunks until we reach ~`PDF_CHUNK_TOKENS` tokens (words as a proxy).  
The **first heading-like line** of each chunk becomes the `topic` label.

**What to look for:**
- Chunks should be coherent (not cut mid-sentence)
- Topics should look like real section headings, not page numbers
- The word count histogram should cluster around 200–450 words

In [ ]:
# ============================================================
PDF_CHUNK_TOKENS   = 400   # ← TUNE THIS: target words per PDF chunk
PDF_MIN_PAGE_WORDS = 20    # ← TUNE THIS: skip near-empty pages (front matter, blank)
# ============================================================

def extract_heading(text: str) -> str:
    """Return the first non-empty, non-numeric line as a heading candidate."""
    for line in text.splitlines():
        line = line.strip()
        # Skip lines that are just numbers (page numbers, equation labels)
        if line and not line.replace(".", "").replace(" ", "").isnumeric():
            return line[:120]  # cap heading length
    return "(no heading)"


def chunk_pdf(doc, pdf_path, chunk_tokens, min_page_words):
    """
    Slide-window merge pages until we hit the token target.
    Returns a list of chunk dicts.
    """
    chunks = []
    stem = Path(pdf_path).stem

    current_text  = ""
    current_start = None   # first page in this chunk
    chunk_index   = 0

    for page_num in range(len(doc)):
        page_text = doc[page_num].get_text().strip()

        # Skip near-empty pages
        if len(page_text.split()) < min_page_words:
            continue

        # Start a new chunk if this is the first page
        if current_start is None:
            current_start = page_num + 1  # 1-based page number

        combined = current_text + "\n" + page_text if current_text else page_text

        if len(combined.split()) >= chunk_tokens and current_text:
            # Save the current chunk and start a new one
            heading = extract_heading(current_text)
            chunks.append({
                "chunk_id":    f"mh_pdf_{stem}_p{current_start:04d}",
                "source_file": Path(pdf_path).name,
                "source_type": "textbook_pdf",
                "page_number": current_start,
                "topic":       heading,
                "lesson_title": stem.replace("-", " ").replace("_", " ").title(),
                "chunk_text":  current_text.strip(),
                "clean_markdown": current_text.strip(),  # same for textbooks
                "timestamp_start": None,
                "timestamp_end":   None,
                "speaker_id":  None,
            })
            chunk_index += 1
            current_text  = page_text
            current_start = page_num + 1
        else:
            current_text  = combined

    # Save the last chunk
    if current_text and current_start is not None:
        heading = extract_heading(current_text)
        chunks.append({
            "chunk_id":    f"mh_pdf_{stem}_p{current_start:04d}",
            "source_file": Path(pdf_path).name,
            "source_type": "textbook_pdf",
            "page_number": current_start,
            "topic":       heading,
            "lesson_title": stem.replace("-", " ").replace("_", " ").title(),
            "chunk_text":  current_text.strip(),
            "clean_markdown": current_text.strip(),
            "timestamp_start": None,
            "timestamp_end":   None,
            "speaker_id":  None,
        })

    return chunks


pdf_chunks = chunk_pdf(doc, PDF_FILE, PDF_CHUNK_TOKENS, PDF_MIN_PAGE_WORDS)

print(f"Pages in PDF        : {len(doc)}")
print(f"PDF chunks produced : {len(pdf_chunks)}")
print()
print("Sample chunks:")
for c in pdf_chunks[:3]:
    wc = len(c["chunk_text"].split())
    print(f"  {c['chunk_id']:<30}  p.{c['page_number']:<5}  {wc:>4} words  topic: {c['topic'][:60]}")

In [ ]:
# --- Histogram of PDF chunk word counts ---
pdf_wc = [len(c["chunk_text"].split()) for c in pdf_chunks]

plt.figure(figsize=(9, 4))
plt.hist(pdf_wc, bins=30, color="#55A868", edgecolor="white")
plt.axvline(PDF_CHUNK_TOKENS, color="red", linestyle="--", label=f"PDF_CHUNK_TOKENS={PDF_CHUNK_TOKENS}")
plt.xlabel("Words per chunk")
plt.ylabel("Count")
plt.title(f"PDF chunk size distribution — {PDF_FILE.name}")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Median chunk length : {int(np.median(pdf_wc))} words")
print(f"Mean chunk length   : {int(np.mean(pdf_wc))} words")

In [ ]:
# Process the second textbook as well
PDF_FILE_2 = CORPUS_DIR / "deep-learning-2017-mit.pdf"   # ← adjust name if needed
if PDF_FILE_2.exists():
    doc2 = fitz.open(PDF_FILE_2)
    pdf_chunks_2 = chunk_pdf(doc2, PDF_FILE_2, PDF_CHUNK_TOKENS, PDF_MIN_PAGE_WORDS)
    print(f"{PDF_FILE_2.name}: {len(doc2)} pages → {len(pdf_chunks_2)} chunks")
    pdf_chunks.extend(pdf_chunks_2)
    doc2.close()
else:
    print(f"{PDF_FILE_2.name} not found — skipping")

print(f"\nTotal PDF chunks from all textbooks: {len(pdf_chunks)}")

---
## Section 7 — What Are Embeddings?

An **embedding** is a list of numbers (a vector) that represents the *meaning* of a piece of text.  
Two texts about the same topic will produce *similar* vectors; unrelated texts will produce *different* vectors.

We measure similarity using **cosine similarity** — a value between -1 and 1:  
- **1.0** = identical meaning  
- **0.7–0.9** = closely related  
- **0.5–0.7** = somewhat related  
- **< 0.5** = different topics

We use Google's **`gemini-embedding-2`** model, which outputs 768 numbers per text.  
This is the same model Ben used to index the course slides — so our vectors are in the same "language".  
That matters because we'll query Ben's indexed vectors to align transcript topics.

> The embedding step requires a GEMINI_API_KEY and makes API calls.  
> Each call costs a small amount of API quota.  
> We embed a few examples here, and batch-embed all chunks in Section 8.

In [ ]:
# --- Set up the Gemini embedding client ---
from google import genai
from google.genai import types as genai_types

EMBED_MODEL = "gemini-embedding-2"   # ← must match the model used by Ben's index
EMBED_DIM   = 768

_genai_client = genai.Client(api_key=GEMINI_API_KEY)

def embed_text(text: str, task: str = "RETRIEVAL_DOCUMENT") -> list[float]:
    """
    Embed a single text using Gemini.
    task: 'RETRIEVAL_DOCUMENT' for indexing chunks
          'RETRIEVAL_QUERY'    for embedding queries
    """
    result = _genai_client.models.embed_content(
        model=EMBED_MODEL,
        contents=text,
        config=genai_types.EmbedContentConfig(
            task_type=task,
            output_dimensionality=EMBED_DIM,
        ),
    )
    return list(result.embeddings[0].values)


def embed_batch(texts: list[str], task: str = "RETRIEVAL_DOCUMENT",
                delay: float = 0.5) -> list[list[float]]:
    """
    Embed multiple texts with a small delay between calls to avoid rate limiting.
    Prints progress every 10 items.
    """
    vectors = []
    for i, text in enumerate(texts):
        if i > 0 and i % 10 == 0:
            print(f"  Embedded {i}/{len(texts)}...", end="\r")
        vectors.append(embed_text(text, task))
        time.sleep(delay)
    print(f"  Embedded {len(texts)}/{len(texts)} ✓")
    return vectors


print("Embedding client ready.")
print()

# --- Quick demo: embed two sentences and compare ---
text_a = "The central limit theorem states that the distribution of sample means approaches normal."
text_b = "When you average many samples, the result follows a bell curve."
text_c = "The LSTM forget gate controls which information is discarded from memory."

vec_a = embed_text(text_a)
vec_b = embed_text(text_b)
vec_c = embed_text(text_c, task="RETRIEVAL_QUERY")

def cosine_sim(v1, v2):
    v1, v2 = np.array(v1), np.array(v2)
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

print(f"Vector dimension : {len(vec_a)} (expected {EMBED_DIM})")
print(f"First 5 values   : {[round(v, 4) for v in vec_a[:5]]}")
print()
print(f"Similarity: A vs B (same topic, different wording) : {cosine_sim(vec_a, vec_b):.4f}")
print(f"Similarity: A vs C (different topics)              : {cosine_sim(vec_a, vec_c):.4f}")
print()
print("Expected: A-B score >> A-C score (A and B are about CLT, C is about LSTM)")

---
## Section 8 — Topic Alignment

The transcript has no module labels — we don't know if a given turn is about module 3.1 or 3.7.  
**Alignment** solves this by finding the *closest course slide* for each transcript turn.

How it works:
1. Embed the transcript turn (768-dim vector)
2. Query Ben's table `rag_chunks_ben0602` for the 1 nearest slide chunk
3. Inherit the slide's `source_file`, `lesson_title`, `topic` as metadata for the transcript turn
4. Store the `alignment_score` (cosine similarity) — used for quality review

Why this is useful:
- Retrieval can later filter by module (e.g. "only show chunks from 3.7")
- The student sees a source label ("Module 3.1 — Statistics") next to the transcript citation
- We can spot-check alignment quality in Section 9

> This section makes one Gemini API call per transcript turn AND one DB query per turn.  
> For ~100 turns it takes about 2–3 minutes.  
> **Skip this section** if you don't have DB access — turns will be labelled `topic="unaligned"`.

In [ ]:
# --- DB alignment (requires DATABASE_URL) ---
import asyncio
import asyncpg

ALIGNMENT_TOP_K = 1   # we only need the best match

async def align_chunks(turn_texts: list[str]) -> list[dict]:
    """
    For each transcript turn:
      1. Embed with Gemini (RETRIEVAL_QUERY task — we are querying Ben's index)
      2. Find nearest chunk in rag_chunks_ben0602
      3. Return alignment metadata
    """
    conn = await asyncpg.connect(DATABASE_URL)
    results = []

    for i, text in enumerate(turn_texts):
        if i % 10 == 0:
            print(f"  Aligning turn {i+1}/{len(turn_texts)}...", end="\r")

        # Embed as a query (not a document — we're searching Ben's index)
        vec = embed_text(text, task="RETRIEVAL_QUERY")
        vec_str = "[" + ",".join(str(v) for v in vec) + "]"

        row = await conn.fetchrow("""
            SELECT
                chunk_id,
                source_file,
                COALESCE(lesson_title, '') AS lesson_title,
                COALESCE(topic, '')        AS topic,
                REGEXP_REPLACE(source_file, '.*?([0-9]+\\.[0-9]+).*', '\\1', '') AS mod,
                1 - (embedding <=> CAST($1 AS vector)) AS score
            FROM rag_chunks_ben0602
            ORDER BY embedding <=> CAST($1 AS vector)
            LIMIT 1
        """, vec_str)

        if row:
            results.append({
                "aligned_chunk_id": row["chunk_id"],
                "topic":            row["topic"] or row["lesson_title"],
                "lesson_title":     row["lesson_title"],
                "source_aligned":   row["source_file"],
                "mod":              row["mod"],
                "alignment_score":  float(row["score"]),
            })
        else:
            results.append({
                "aligned_chunk_id": None,
                "topic":            "unaligned",
                "lesson_title":     "unknown",
                "source_aligned":   None,
                "mod":              "?",
                "alignment_score":  0.0,
            })

        time.sleep(0.3)   # gentle rate-limit on Gemini API

    await conn.close()
    print(f"  Alignment done: {len(results)} turns processed ✓")
    return results


if DATABASE_URL:
    print(f"Aligning {len(turns)} transcript turns against rag_chunks_ben0602 ...")
    print("(This takes ~2-3 minutes for ~100 turns — Gemini API calls)")
    alignment_results = asyncio.run(align_chunks([t["text"] for t in turns]))

    # Write alignment back into each turn
    for turn, align in zip(turns, alignment_results):
        turn.update(align)

    print("Alignment complete.")
else:
    print("DATABASE_URL not set — skipping alignment.")
    print("Turns will be labelled topic='unaligned'.")
    for turn in turns:
        turn["aligned_chunk_id"] = None
        turn["topic"]            = "unaligned"
        turn["lesson_title"]     = "unknown"
        turn["alignment_score"] = 0.0
        turn["mod"]             = "?"

---
## Section 9 — Quality Review

Before writing to the database, review the alignment results.  

**What to check:**
1. **Score distribution** — most scores should be above 0.60. Below 0.50 = poor alignment (off-topic or unrelated slide).
2. **Module spread** — chunks should spread across modules. If everything maps to one module, the corpus may be unbalanced.
3. **Low-score outliers** — read these turns. Decide: keep them as "general" context, or drop them?
4. **High-score examples** — confirm the alignment makes sense (turn about CLT → matched to stats module, not LSTM module).

In [ ]:
# --- Alignment score distribution ---
if DATABASE_URL:
    scores = [t["alignment_score"] for t in turns]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Left: score histogram
    axes[0].hist(scores, bins=25, color="#4C72B0", edgecolor="white")
    axes[0].axvline(0.60, color="orange", linestyle="--", label="threshold=0.60")
    axes[0].axvline(0.50, color="red",    linestyle="--", label="low=0.50")
    axes[0].set_xlabel("Alignment score (cosine similarity)")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Transcript → slide alignment scores")
    axes[0].legend()

    # Right: module distribution
    mod_counts = defaultdict(int)
    for t in turns:
        mod_counts[t.get("mod", "?")] += 1
    mods = sorted(mod_counts)
    axes[1].bar(mods, [mod_counts[m] for m in mods], color="#DD8452")
    axes[1].set_xlabel("Module")
    axes[1].set_ylabel("Transcript chunks")
    axes[1].set_title("Transcript chunks by module")
    plt.xticks(rotation=45, ha="right")

    plt.suptitle(f"Alignment quality — {TRANSCRIPT_FILE.name}")
    plt.tight_layout()
    plt.show()

    print(f"Mean alignment score  : {np.mean(scores):.3f}")
    print(f"Median alignment score: {np.median(scores):.3f}")
    low = sum(1 for s in scores if s < 0.50)
    print(f"Low-confidence turns  : {low} (score < 0.50) — review below")
else:
    print("Skipped (no DB connection)")

In [ ]:
# --- Review low-scoring turns ---
# These are turns where the transcript didn't align well to any slide.
# Read them and decide: keep (they may still be useful) or filter them out.

if DATABASE_URL:
    LOW_SCORE_THRESHOLD = 0.50   # ← TUNE THIS
    low_turns = [t for t in turns if t["alignment_score"] < LOW_SCORE_THRESHOLD]
    print(f"Turns with score < {LOW_SCORE_THRESHOLD}: {len(low_turns)}")
    print()
    for t in sorted(low_turns, key=lambda x: x["alignment_score"])[:8]:
        print(f"  score={t['alignment_score']:.3f}  mod={t.get('mod','?')}  [{t['source_type']}]")
        print(f"  text: {t['text'].strip()[:200]}")
        print()
else:
    print("Skipped (no DB connection)")

In [ ]:
# --- Review high-scoring turns (alignment sanity check) ---
# The top-scoring turns should clearly be about the topic they were matched to.

if DATABASE_URL:
    high_turns = sorted(turns, key=lambda x: x["alignment_score"], reverse=True)[:5]
    print("Top-aligned transcript turns:")
    print()
    for t in high_turns:
        print(f"  score={t['alignment_score']:.3f}  mod={t.get('mod','?')}")
        print(f"  matched slide  : {t.get('source_aligned', 'N/A')}")
        print(f"  topic          : {t.get('topic', 'N/A')}")
        print(f"  transcript text: {t['text'].strip()[:200]}")
        print()
else:
    print("Skipped (no DB connection)")

---
## Section 10 — Export Approved Chunks

Once you are happy with the chunking parameters and alignment quality, export everything to a JSON file.  
This file is the input to the **Phase 2 upsert script** which will embed all chunks and write them to the database.

**No database writes happen in this notebook.**  
The export is just a preview file you can inspect, share, and commit for review.

In [ ]:
# --- Assemble all chunks ---

# Convert transcript turns to the standard chunk schema
transcript_chunk_dicts = []
for t in turns:
    transcript_chunk_dicts.append({
        "chunk_id":        t["chunk_id"],
        "source_file":     t["source_file"],
        "source_type":     t["source_type"],
        "lesson_title":    t.get("lesson_title", ""),
        "topic":           t.get("topic", "unaligned"),
        "page_number":     None,
        "timestamp_start": t["start"],
        "timestamp_end":   t["end"],
        "speaker_id":      t["speaker"],
        "chunk_text":      t["text"].strip(),
        "clean_markdown":  None,
        "aligned_chunk_id": t.get("aligned_chunk_id"),
        "alignment_score":  t.get("alignment_score", 0.0),
        "token_estimate":   len(t["text"].split()),
        "metadata": {
            "source": TRANSCRIPT_FILE.name,
            "instructor": INSTRUCTOR,
            "chunk_max_words": CHUNK_MAX_WORDS,
            "min_turn_words": MIN_TURN_WORDS,
        },
    })

# PDF chunks already have the right schema
pdf_chunk_dicts = []
for c in pdf_chunks:
    c_copy = dict(c)
    c_copy["aligned_chunk_id"] = None
    c_copy["alignment_score"]  = None
    c_copy["token_estimate"]   = len(c["chunk_text"].split())
    c_copy["metadata"] = {
        "source": c["source_file"],
        "pdf_chunk_tokens": PDF_CHUNK_TOKENS,
    }
    pdf_chunk_dicts.append(c_copy)

all_chunks = transcript_chunk_dicts + pdf_chunk_dicts

# --- Summary ---
from collections import Counter
type_summary = Counter(c["source_type"] for c in all_chunks)
print("=" * 55)
print("EXPORT SUMMARY")
print("=" * 55)
for k, v in sorted(type_summary.items()):
    print(f"  {k:<35}: {v:>5} chunks")
print(f"  {'TOTAL':<35}: {len(all_chunks):>5} chunks")
print()

if DATABASE_URL:
    tr_scores = [c["alignment_score"] for c in transcript_chunk_dicts if c["alignment_score"]]
    if tr_scores:
        print(f"  Transcript alignment: mean={np.mean(tr_scores):.3f}, median={np.median(tr_scores):.3f}")
        print(f"  Low-confidence (< 0.50): {sum(1 for s in tr_scores if s < 0.50)} turns")

print()
print("Parameters used:")
print(f"  CHUNK_MAX_WORDS  = {CHUNK_MAX_WORDS}")
print(f"  MIN_TURN_WORDS   = {MIN_TURN_WORDS}")
print(f"  PDF_CHUNK_TOKENS = {PDF_CHUNK_TOKENS}")
print(f"  EMBED_MODEL      = {EMBED_MODEL}")

In [ ]:
# --- Save to JSON ---
OUTPUT_FILE = OUTPUT_DIR / "chunks_preview.json"

with open(OUTPUT_FILE, "w") as f:
    json.dump({
        "metadata": {
            "generated_by": "rag_ingestion_dev.ipynb",
            "total_chunks": len(all_chunks),
            "type_counts": dict(type_summary),
            "parameters": {
                "CHUNK_MAX_WORDS":  CHUNK_MAX_WORDS,
                "MIN_TURN_WORDS":   MIN_TURN_WORDS,
                "PDF_CHUNK_TOKENS": PDF_CHUNK_TOKENS,
                "embed_model":      EMBED_MODEL,
            },
        },
        "chunks": all_chunks,
    }, f, indent=2, default=str)

print(f"Saved {len(all_chunks)} chunks to: {OUTPUT_FILE}")
print()
print("Next step: review chunks_preview.json, then run the Phase 2 upsert script")
print("to embed and write the chunks to the rag_chunks_menghai table.")